In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rpghost/scaffoldsplit/val.csv
/kaggle/input/datasets/rpghost/scaffoldsplit/train.csv
/kaggle/input/datasets/rpghost/scaffoldsplit/test.csv


In [6]:
# !pip install -q chemprop

In [7]:
import torch
import chemprop

print("Chemprop version:", chemprop.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

!chemprop train --help

Chemprop version: 2.3.1
PyTorch version: 2.11.0+cu128
CUDA available: True
usage: chemprop train [-h] [--logfile [LOGFILE]] [-v] [-q]
                      [-s SMILES_COLUMNS [SMILES_COLUMNS ...]]
                      [-r REACTION_COLUMNS [REACTION_COLUMNS ...]]
                      [--no-header-row] [-n NUM_WORKERS] [-b BATCH_SIZE]
                      [--accelerator ACCELERATOR] [--devices DEVICES]
                      [--rxn-mode {REAC_PROD,REAC_PROD_BALANCE,REAC_DIFF,REAC_DIFF_BALANCE,PROD_DIFF,PROD_DIFF_BALANCE}]
                      [--multi-hot-atom-featurizer-mode {V1,V2,ORGANIC,RIGR}]
                      [--keep-h] [--add-h] [--ignore-stereo] [--reorder-atoms]
                      [--molecule-featurizers {morgan_binary,morgan_count,rdkit_2d,v1_rdkit_2d,v1_rdkit_2d_normalized,charge} [{morgan_binary,morgan_count,rdkit_2d,v1_rdkit_2d,v1_rdkit_2d_normalized,charge} ...]]
                      [--descriptors-path DESCRIPTORS_PATH]
                      [--descriptors-colum

In [8]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/rpghost/scaffoldsplit/val.csv
/kaggle/input/datasets/rpghost/scaffoldsplit/train.csv
/kaggle/input/datasets/rpghost/scaffoldsplit/test.csv


In [9]:
import pandas as pd

train_path = "/kaggle/input/datasets/rpghost/scaffoldsplit/train.csv"
val_path   = "/kaggle/input/datasets/rpghost/scaffoldsplit/val.csv"
test_path  = "/kaggle/input/datasets/rpghost/scaffoldsplit/test.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}: {df.shape}")
    print(df[["smiles", "hERG_blocker"]].head())
    print(df["hERG_blocker"].value_counts())


Train: (7451, 4)
                                              smiles  hERG_blocker
0  CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...             1
1               CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1             0
2  N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...             1
3             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1             1
4      CC(C)CN(C(=O)c1cccc(C(F)(F)F)c1Cl)[C@H]1CCNC1             1
hERG_blocker
1    4932
0    2519
Name: count, dtype: int64

Validation: (954, 4)
                                              smiles  hERG_blocker
0        O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1             0
1            CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1             1
2  Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(Br)cc5)C4)n...             1
3  Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(C#N)cc5)C4)...             1
4  COc1ccc(C23CC2CN(CCCSc2nnc(-c4cccc5nc(C)ccc45)...             1
hERG_blocker
1    663
0    291
Name: count, dtype: int64

Test: (1944, 4)
                       

In [10]:
import os
import pandas as pd

# Check expected columns and class distributions
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}: {df.shape}")

    assert "smiles" in df.columns, f"{name}: missing smiles column"
    assert "hERG_blocker" in df.columns, f"{name}: missing target column"

    print(df["hERG_blocker"].value_counts(dropna=False))
    print("Missing SMILES:", df["smiles"].isna().sum())
    print("Missing labels:", df["hERG_blocker"].isna().sum())

    assert df["smiles"].notna().all()
    assert df["hERG_blocker"].notna().all()
    assert set(df["hERG_blocker"].unique()).issubset({0, 1})


Train: (7451, 4)
hERG_blocker
1    4932
0    2519
Name: count, dtype: int64
Missing SMILES: 0
Missing labels: 0

Validation: (954, 4)
hERG_blocker
1    663
0    291
Name: count, dtype: int64
Missing SMILES: 0
Missing labels: 0

Test: (1944, 4)
hERG_blocker
1    1329
0     615
Name: count, dtype: int64
Missing SMILES: 0
Missing labels: 0


In [11]:
!chemprop --version
!ls -lh /kaggle/working/chemprop_data/
!nvidia-smi

chemprop 2.3.1
ls: cannot access '/kaggle/working/chemprop_data/': No such file or directory
Tue Oct 13 15:26:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |    

In [ ]:
!chemprop train \
    -i /kaggle/input/datasets/rpghost/scaffoldsplit/train.csv \
       /kaggle/input/datasets/rpghost/scaffoldsplit/val.csv \
       /kaggle/input/datasets/rpghost/scaffoldsplit/test.csv \
    -s smiles \
    --target-columns hERG_blocker \
    -t classification \
    --metrics prc roc binary-mcc \
    --epochs 50 \
    --patience 7 \
    -b 64 \
    --accelerator gpu \
    --devices 1 \
    --pytorch-seed 123 \
    -o /kaggle/working/chemprop_model

2026-10-13T15:28:48 - INFO:chemprop.cli.main - Running in mode 'train' with args: {'smiles_columns': ['smiles'], 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'gpu', 'devices': '1', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'config_path': None, 'data_path': [PosixPath('/kaggle/input/datasets/rpghost/scaffoldsplit/train.csv'), PosixPath('/kaggle/input/datasets/rpg

In [18]:
for root, dirs, files in os.walk(
    "/kaggle/working/chemprop_model"
):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/chemprop_model/config.toml
/kaggle/working/chemprop_model/model_0/best.pt
/kaggle/working/chemprop_model/model_0/test_predictions.csv
/kaggle/working/chemprop_model/model_0/checkpoints/last.ckpt
/kaggle/working/chemprop_model/model_0/checkpoints/best-epoch=13-val_loss=0.49.ckpt
/kaggle/working/chemprop_model/model_0/trainer_logs/version_0/events.out.tfevents.1791905354.e067274a66cc.310.0
/kaggle/working/chemprop_model/model_0/trainer_logs/version_0/hparams.yaml


In [19]:
import pandas as pd

pred_path = "/kaggle/working/chemprop_model/model_0/test_predictions.csv"

pred_df = pd.read_csv(pred_path)

print("Shape:", pred_df.shape)
print("Columns:", pred_df.columns.tolist())
display(pred_df.head(10))
print(pred_df.dtypes)

Shape: (1944, 2)
Columns: ['smiles', 'hERG_blocker']


,smiles,hERG_blocker
0,CN(C)CC/C=C1/c2ccccc2COc2ccccc21,0.896912
1,Cc1cnc(-c2nnc(SCCCN3C[C@@H]4C[C@]4(c4ccc(C(F)(...,0.989175
2,Cn1c(SCCCN2C[C@@H]3C[C@]3(c3ccc(C(F)(F)F)cc3)C...,0.996394
3,Cc1cc(-c2ccc3cc(CCN4CCC[C@H]4C)ccc3n2)nc(-c2cc...,0.911355
4,CC1(C)CC(NC(=O)CCO)c2cc(-c3ccc(Cl)cc3)c(-c3ccc...,0.918841
5,C[C@@H]1CCCN1CCc1ccc2nc(-c3nc4ccccc4s3)ccc2c1,0.799923
6,C[C@@H](O)C(=O)NC1CC(C)(C)Oc2nc(-c3ccc(Cl)cc3C...,0.873378
7,c1ccc(C(Cc2ccccc2OCC2CC2)N2CCNCC2)cc1,0.543187
8,Cn1nccc1-c1cc(NC(=O)c2ccc(F)cc2F)ccc1OCCN1CCOCC1,0.530861
9,C[C@@H]1CCCN1CCc1ccc2nc(-c3csc(-c4cc(Cl)nc(Cl)...,0.878022


smiles           object
hERG_blocker    float64
dtype: object


In [20]:
!cat /kaggle/working/chemprop_model/config.toml

smiles-columns = [smiles]
batch-size = 64
accelerator = gpu
devices = 1
data-path = [/kaggle/input/datasets/rpghost/scaffoldsplit/train.csv, /kaggle/input/datasets/rpghost/scaffoldsplit/val.csv, /kaggle/input/datasets/rpghost/scaffoldsplit/test.csv]
output-dir = /kaggle/working/chemprop_model
target-columns = [hERG_blocker]
task-type = classification
metrics = [prc, roc, binary-mcc]
epochs = 50
patience = 7
pytorch-seed = 123
num-workers = 0
rxn-mode = REAC_DIFF
multi-hot-atom-featurizer-mode = V2
frzn-ffn-layers = 0
ensemble-size = 1
message-hidden-dim = [300]
depth = [3]
dropout = 0.0
aggregation = norm
aggregation-norm = 100
activation = RELU
ffn-hidden-dim = [300]
ffn-num-layers = 1
multiclass-num-classes = 3
atom-ffn-hidden-dim = [300]
atom-ffn-num-layers = 1
atom-multiclass-num-classes = 3
bond-ffn-hidden-dim = [300]
bond-ffn-num-layers = 1
bond-multiclass-num-classes = 3
atom-constrainer-ffn-hidden-dim = [300]
atom-constrainer-ffn-num-layers = 1
bond-constrainer-ffn-hidden-dim =

In [21]:
import pandas as pd
import os

test_pred_path = (
    "/kaggle/working/chemprop_model/model_0/test_predictions.csv"
)

pred_df = pd.read_csv(test_pred_path)

print("Prediction file shape:", pred_df.shape)
print("Columns:", pred_df.columns.tolist())
display(pred_df.head(10))

print("\nData types:")
print(pred_df.dtypes)

Prediction file shape: (1944, 2)
Columns: ['smiles', 'hERG_blocker']


,smiles,hERG_blocker
0,CN(C)CC/C=C1/c2ccccc2COc2ccccc21,0.896912
1,Cc1cnc(-c2nnc(SCCCN3C[C@@H]4C[C@]4(c4ccc(C(F)(...,0.989175
2,Cn1c(SCCCN2C[C@@H]3C[C@]3(c3ccc(C(F)(F)F)cc3)C...,0.996394
3,Cc1cc(-c2ccc3cc(CCN4CCC[C@H]4C)ccc3n2)nc(-c2cc...,0.911355
4,CC1(C)CC(NC(=O)CCO)c2cc(-c3ccc(Cl)cc3)c(-c3ccc...,0.918841
5,C[C@@H]1CCCN1CCc1ccc2nc(-c3nc4ccccc4s3)ccc2c1,0.799923
6,C[C@@H](O)C(=O)NC1CC(C)(C)Oc2nc(-c3ccc(Cl)cc3C...,0.873378
7,c1ccc(C(Cc2ccccc2OCC2CC2)N2CCNCC2)cc1,0.543187
8,Cn1nccc1-c1cc(NC(=O)c2ccc(F)cc2F)ccc1OCCN1CCOCC1,0.530861
9,C[C@@H]1CCCN1CCc1ccc2nc(-c3csc(-c4cc(Cl)nc(Cl)...,0.878022



Data types:
smiles           object
hERG_blocker    float64
dtype: object


In [22]:
!chemprop predict --help

usage: chemprop predict [-h] [--logfile [LOGFILE]] [-v] [-q]
                        [-s SMILES_COLUMNS [SMILES_COLUMNS ...]]
                        [-r REACTION_COLUMNS [REACTION_COLUMNS ...]]
                        [--no-header-row] [-n NUM_WORKERS] [-b BATCH_SIZE]
                        [--accelerator ACCELERATOR] [--devices DEVICES]
                        [--rxn-mode {REAC_PROD,REAC_PROD_BALANCE,REAC_DIFF,REAC_DIFF_BALANCE,PROD_DIFF,PROD_DIFF_BALANCE}]
                        [--multi-hot-atom-featurizer-mode {V1,V2,ORGANIC,RIGR}]
                        [--keep-h] [--add-h] [--ignore-stereo]
                        [--reorder-atoms]
                        [--molecule-featurizers {morgan_binary,morgan_count,rdkit_2d,v1_rdkit_2d,v1_rdkit_2d_normalized,charge} [{morgan_binary,morgan_count,rdkit_2d,v1_rdkit_2d,v1_rdkit_2d_normalized,charge} ...]]
                        [--descriptors-path DESCRIPTORS_PATH]
                        [--descriptors-columns DESCRIPTORS_COLUMNS [DESCR

In [26]:
!chemprop predict \
    -i /kaggle/input/datasets/rpghost/scaffoldsplit/val.csv \
    -o /kaggle/working/chemprop_model/val_predictions.csv \
    -s smiles \
    --model-paths /kaggle/working/chemprop_model/model_0/best.pt \
    --accelerator gpu \
    --devices 1

2026-10-13T15:41:52 - INFO:chemprop.cli.main - Running in mode 'predict' with args: {'smiles_columns': ['smiles'], 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'gpu', 'devices': '1', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'test_path': PosixPath('/kaggle/input/datasets/rpghost/scaffoldsplit/val.csv'), 'output': PosixPath('/kaggle/working/chemprop_model/val_pre

In [27]:
import pandas as pd

val_pred_path = (
    "/kaggle/working/chemprop_model/val_predictions.csv"
)

val_pred_df = pd.read_csv(val_pred_path)

print("Shape:", val_pred_df.shape)
print("Columns:", val_pred_df.columns.tolist())
display(val_pred_df.head(10))

Shape: (954, 4)
Columns: ['Unnamed: 0', 'inchi_key', 'smiles', 'hERG_blocker']


,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,0,ZKLPARSLTMPFCP-UHFFFAOYSA-N,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0.478434
1,4,UIEATEWHFDRYRU-UHFFFAOYSA-N,CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1,0.832980
2,22,VUFRFPUOFHXBBM-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(Br)cc5)C4)n...,0.991788
3,23,CLLQEXUCEWAACY-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(C#N)cc5)C4)...,0.987053
4,24,NKIUMOVWDJZMRB-UHFFFAOYSA-N,COc1ccc(C23CC2CN(CCCSc2nnc(-c4cccc5nc(C)ccc45)...,0.980758
5,26,CCJLDYTVWAENSM-NZQKXSOJSA-N,Cc1ccc(-c2nnc(SCCCN3C[C@@H]4C[C@]4(c4ccc(C(F)(...,0.993998
6,37,AAONJNLMLWRZJE-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(Cl)cc5)C4)n...,0.987742
7,39,VUFRFPUOFHXBBM-HRFSGMKKSA-N,Cc1ccc2c(-c3nnc(SCCCN4C[C@H]5C[C@@]5(c5ccc(Br)...,0.989305
8,40,VUFRFPUOFHXBBM-CCLHPLFOSA-N,Cc1ccc2c(-c3nnc(SCCCN4C[C@@H]5C[C@]5(c5ccc(Br)...,0.996976
9,42,AWJOAMPQOUEKCN-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(C(F)(F)F)cc...,0.989611


In [37]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, matthews_corrcoef


val_true = pd.read_csv(
    "/kaggle/input/datasets/rpghost/scaffoldsplit/val.csv"
)
val_pred = pd.read_csv(
    "/kaggle/working/chemprop_model/val_predictions.csv"
)

# Confirm prediction rows match the validation input order.
assert len(val_true) == len(val_pred) == 954
assert (
    val_true["smiles"].astype(str).to_numpy()
    == val_pred["smiles"].astype(str).to_numpy()
).all(), "SMILES order mismatch!"

y_val = val_true["hERG_blocker"].astype(int).to_numpy()
p_val = val_pred["hERG_blocker"].astype(float).to_numpy()

assert np.isfinite(p_val).all()
assert ((p_val >= 0) & (p_val <= 1)).all()

# target_sensitivity = 0.90
# results = []

# for threshold in np.unique(p_val):
#     y_hat = (p_val >= threshold).astype(int)
#     tn, fp, fn, tp = confusion_matrix(
#         y_val, y_hat, labels=[0, 1]
#     ).ravel()

#     sensitivity = tp / (tp + fn)
#     specificity = tn / (tn + fp)

#     if sensitivity >= target_sensitivity:
#         results.append({
#             "threshold": threshold,
#             "sensitivity": sensitivity,
#             "specificity": specificity,
#             "TN": tn, "FP": fp, "FN": fn, "TP": tp
#         })

# if results:
#     threshold_df = pd.DataFrame(results)
#     best = threshold_df.sort_values(
#         ["specificity", "threshold"],
#         ascending=[False, False]
#     ).iloc[0]

#     best_threshold = float(best["threshold"])

#     print("Selected validation threshold:", best_threshold)
#     print(f"Sensitivity: {best['sensitivity']:.4f}")
#     print(f"Specificity: {best['specificity']:.4f}")
#     print(
#         f"TN={int(best['TN'])}, FP={int(best['FP'])}, "
#         f"FN={int(best['FN'])}, TP={int(best['TP'])}"
#     )
# else:
#     best_threshold = None
#     print("No threshold achieves 95% validation sensitivity.")



target_sensitivity = 0.90
results = []

for threshold in np.unique(p_val):
    y_hat = (p_val >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_hat, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    mcc = matthews_corrcoef(y_val, y_hat)

    results.append({
        "threshold": threshold,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "MCC": mcc,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })

results_df = pd.DataFrame(results)

# Strategy A: highest MCC on validation
best_mcc = results_df.loc[results_df["MCC"].idxmax()]

# Strategy B: highest specificity subject to >=90% sensitivity
eligible = results_df[
    results_df["sensitivity"] >= target_sensitivity
]

best_sensitivity = (
    eligible.sort_values(
        ["specificity", "threshold"],
        ascending=[False, False]
    ).iloc[0]
    if not eligible.empty else None
)

print("===== MAXIMUM MCC THRESHOLD =====")
print(best_mcc.to_string())

print("\n===== 90% SENSITIVITY THRESHOLD =====")
if best_sensitivity is not None:
    print(best_sensitivity.to_string())
else:
    print("No threshold achieves the target sensitivity.")

===== MAXIMUM MCC THRESHOLD =====
threshold        0.642339
sensitivity      0.758673
specificity      0.707904
MCC              0.441771
TN             206.000000
FP              85.000000
FN             160.000000
TP             503.000000

===== 90% SENSITIVITY THRESHOLD =====
threshold        0.410295
sensitivity      0.900452
specificity      0.381443
MCC              0.333883
TN             111.000000
FP             180.000000
FN              66.000000
TP             597.000000


In [38]:
# Store both selected thresholds
threshold_mcc = float(best_mcc["threshold"])

threshold_sensitivity = (
    float(best_sensitivity["threshold"])
    if best_sensitivity is not None
    else None
)

print("\nThreshold maximizing MCC:", threshold_mcc)
print("Threshold targeting 90% sensitivity:", threshold_sensitivity)


Threshold maximizing MCC: 0.64233875
Threshold targeting 90% sensitivity: 0.41029468


In [39]:
# Evaluate the maximum-MCC strategy
best_threshold = threshold_mcc

# Evaluate the 90%-sensitivity strategy
# best_threshold = threshold_sensitivity


In [40]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix
)

if best_threshold is None:
    raise ValueError(
        "No threshold met the sensitivity target. "
        "Inspect validation results before choosing another target."
    )

test_true = pd.read_csv(
    "/kaggle/input/datasets/rpghost/scaffoldsplit/test.csv"
)
test_pred = pd.read_csv(
    "/kaggle/working/chemprop_model/model_0/test_predictions.csv"
)

assert len(test_true) == len(test_pred) == 1944
assert (
    test_true["smiles"].astype(str).to_numpy()
    == test_pred["smiles"].astype(str).to_numpy()
).all(), "SMILES order mismatch!"

y_test = test_true["hERG_blocker"].astype(int).to_numpy()
p_test = test_pred["hERG_blocker"].astype(float).to_numpy()

y_test_hat = (p_test >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, y_test_hat, labels=[0, 1]
).ravel()

metrics = {
    "Threshold (selected on validation)": best_threshold,
    "Accuracy": accuracy_score(y_test, y_test_hat),
    "Precision": precision_score(y_test, y_test_hat, zero_division=0),
    "Sensitivity / Recall": recall_score(y_test, y_test_hat, zero_division=0),
    "Specificity": tn / (tn + fp),
    "F1-score": f1_score(y_test, y_test_hat, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, p_test),
    "PR-AUC (Average Precision)": average_precision_score(y_test, p_test),
    "MCC": matthews_corrcoef(y_test, y_test_hat),
}

print("===== CHEMPROP TEST RESULTS =====")
for name, value in metrics.items():
    print(f"{name:40s}: {value:.4f}")

print("\n===== CONFUSION MATRIX =====")
print("                 Predicted 0   Predicted 1")
print(f"Actual 0              {tn:6d}        {fp:6d}")
print(f"Actual 1              {fn:6d}        {tp:6d}")

===== CHEMPROP TEST RESULTS =====
Threshold (selected on validation)      : 0.6423
Accuracy                                : 0.7248
Precision                               : 0.8297
Sensitivity / Recall                    : 0.7517
Specificity                             : 0.6667
F1-score                                : 0.7888
ROC-AUC                                 : 0.7792
PR-AUC (Average Precision)              : 0.8771
MCC                                     : 0.4007

===== CONFUSION MATRIX =====
                 Predicted 0   Predicted 1
Actual 0                 410           205
Actual 1                 330           999
